# Improved training, validation tuning, testing, and Grad-CAM

A lighter alternative to PSO/BAT: baseline validation, a small validation-only grid search, final training, one held-out test, and a random Grad-CAM.

> Research software only; Grad-CAM is not clinical lesion segmentation or diagnosis.

## 1. Setup and dataset

Use matching ImageFolder class folders in `train`, `valid`, and `test`. The test set remains untouched until section 6.

In [ ]:
from pathlib import Path
from datetime import datetime
import copy, json, random, sys

ROOT = Path.cwd()
if not (ROOT / 'src').is_dir(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

DATA_DIR = ROOT / 'data' / 'images'
if not DATA_DIR.is_dir(): DATA_DIR = ROOT / 'split_dataset'
OUTPUT_DIR = ROOT / 'outputs'
CHECKPOINT_DIR = ROOT / 'models'
ARCHITECTURE = 'mobilenet_v2'  # or 'resnet18'
IMAGE_SIZE, BATCH_SIZE, WORKERS, SEED = 224, 32, 2, 42
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')

for split in ('train', 'valid', 'test'):
    assert (DATA_DIR / split).is_dir(), f'Missing {DATA_DIR / split}'
print(f'Using dataset: {DATA_DIR}')

## 2. Imports and data loaders

Uses ImageNet normalization, augmentation, weighted sampling, label smoothing, learning-rate reduction, and early stopping.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from PIL import Image
from IPython.display import display

from skin_cancer.data import evaluation_display_transform, evaluation_transform, imagefolder, make_eval_loader, make_train_loader, training_transform
from skin_cancer.gradcam import GradCAM, save_overlay
from skin_cancer.models import build_model, gradcam_layer
from skin_cancer.training import report, run_epoch, save_checkpoint, set_seed

torch.hub.set_dir(str(ROOT / '.cache' / 'torch'))
set_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
train_dataset = imagefolder(DATA_DIR / 'train', training_transform(IMAGE_SIZE))
valid_dataset = imagefolder(DATA_DIR / 'valid', evaluation_transform(IMAGE_SIZE))
test_dataset = imagefolder(DATA_DIR / 'test', evaluation_transform(IMAGE_SIZE))
assert train_dataset.classes == valid_dataset.classes == test_dataset.classes
CLASS_NAMES = train_dataset.classes
print(f'Using {DEVICE}; train counts: {dict(zip(CLASS_NAMES, np.bincount(train_dataset.targets)))}')
train_loader = make_train_loader(train_dataset, BATCH_SIZE, WORKERS)
valid_loader = make_eval_loader(valid_dataset, BATCH_SIZE, WORKERS)
test_loader = make_eval_loader(test_dataset, BATCH_SIZE, WORKERS)

## 3. Training and validation utility

In [ ]:
def fit(config, epochs, patience=5, verbose=True):
    model = build_model(ARCHITECTURE, len(CLASS_NAMES), pretrained=True, dropout=config['dropout']).to(DEVICE)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
    optimizer = AdamW(model.parameters(), lr=config['lr'], weight_decay=config['weight_decay'])
    scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.3, patience=2)
    scaler = torch.amp.GradScaler(DEVICE.type, enabled=DEVICE.type == 'cuda')
    best_f1, stale, best_state, best_metrics, history = -1.0, 0, None, None, []
    for epoch in range(1, epochs + 1):
        train_loss, train_metrics, _, _ = run_epoch(model, train_loader, criterion, DEVICE, optimizer, scaler)
        valid_loss, valid_metrics, _, _ = run_epoch(model, valid_loader, criterion, DEVICE)
        scheduler.step(valid_metrics['macro_f1'])
        row = {'epoch': epoch, 'train_loss': train_loss, 'valid_loss': valid_loss, **{f'train_{k}': v for k, v in train_metrics.items()}, **{f'valid_{k}': v for k, v in valid_metrics.items()}}
        history.append(row)
        if verbose: print(json.dumps({k: round(v, 4) if isinstance(v, float) else v for k, v in row.items()}))
        if valid_metrics['macro_f1'] > best_f1:
            best_f1, stale, best_state, best_metrics = valid_metrics['macro_f1'], 0, copy.deepcopy(model.state_dict()), valid_metrics.copy()
        else: stale += 1
        if stale >= patience:
            if verbose: print('Early stopping')
            break
    model.load_state_dict(best_state)
    return model, best_metrics, pd.DataFrame(history)

## 4. Baseline validation

The baseline is selected by validation macro-F1 only.

In [ ]:
BASELINE = {'lr': 3e-4, 'dropout': 0.25, 'weight_decay': 1e-4}
baseline_model, baseline_valid, baseline_history = fit(BASELINE, epochs=12)
print('Baseline validation:', baseline_valid)
baseline_history.plot(x='epoch', y=['train_macro_f1', 'valid_macro_f1'], title='Baseline macro-F1')
plt.ylim(0, 1); plt.show()

## 5. Lightweight hyperparameter tuning

A small grid search replaces PSO/BAT to keep this notebook faster. It never uses the test split.

In [ ]:
TUNING_EPOCHS = 6
CANDIDATES = [
    {'lr': 1e-4, 'dropout': 0.20, 'weight_decay': 1e-4},
    {'lr': 3e-4, 'dropout': 0.30, 'weight_decay': 1e-4},
    {'lr': 5e-4, 'dropout': 0.35, 'weight_decay': 3e-5},
]
trial_log = []
for index, config in enumerate(CANDIDATES, start=1):
    set_seed(SEED + index)
    model, metrics, _ = fit(config, TUNING_EPOCHS, patience=TUNING_EPOCHS, verbose=False)
    trial_log.append({'trial': index, **config, **metrics})
    print(f'Trial {index}/{len(CANDIDATES)}: validation macro-F1={metrics["macro_f1"]:.4f}, {config}')
    del model
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()
comparison = pd.DataFrame([{'method': 'Baseline', **BASELINE, **baseline_valid}, *trial_log]).sort_values('macro_f1', ascending=False)
display(comparison)
winner = comparison.iloc[0]
selected_config = {'lr': float(winner.lr), 'dropout': float(winner.dropout), 'weight_decay': float(winner.weight_decay)}
selected_method = winner['method'] if pd.notna(winner.get('method')) else f'Grid trial {int(winner.trial)}'
print(f'Selected {selected_method}: {selected_config}')

## 6. Final training, held-out test, and saved weights

The selected setting trains once, then test results are recorded in a uniquely named checkpoint and JSON description.

In [ ]:
final_model, final_valid, final_history = fit(selected_config, epochs=25, patience=6)
test_loss, test_metrics, test_labels, test_probabilities = run_epoch(final_model, test_loader, nn.CrossEntropyLoss(), DEVICE)
print(f'Test loss: {test_loss:.4f}')
print(test_metrics)
print(report(test_labels, test_probabilities, CLASS_NAMES))
checkpoint_name = f"improved_{ARCHITECTURE}_{selected_method.lower().replace(' ', '-')}_val-f1-{final_valid['macro_f1']:.4f}_acc-{final_valid['accuracy']:.4f}_{RUN_ID}.pth"
CHECKPOINT = CHECKPOINT_DIR / checkpoint_name
run_metadata = {'run_id': RUN_ID, 'selected_method': selected_method, 'selected_config': selected_config, 'validation_metrics': final_valid, 'test_metrics': test_metrics, 'test_loss': test_loss}
save_checkpoint(CHECKPOINT, final_model, ARCHITECTURE, CLASS_NAMES, IMAGE_SIZE, final_valid, run_metadata)
description_path = CHECKPOINT.with_suffix('.json')
description_path.write_text(json.dumps(run_metadata, indent=2), encoding='utf-8')
print(f'Checkpoint: {CHECKPOINT}')
print(f'Description: {description_path}')

## 7. Random held-out image and Grad-CAM

Set `IMAGE_TO_INSPECT` to inspect a specific test image; otherwise a fresh random test image is used.

In [ ]:
IMAGE_TO_INSPECT = None
candidates = [path for path in (DATA_DIR / 'test').rglob('*') if path.suffix.lower() in {'.jpg', '.jpeg', '.png'}]
image_path = Path(IMAGE_TO_INSPECT) if IMAGE_TO_INSPECT else random.choice(candidates)
original = Image.open(image_path).convert('RGB')
display_image = evaluation_display_transform(IMAGE_SIZE)(original)
tensor = evaluation_transform(IMAGE_SIZE)(original).unsqueeze(0).to(DEVICE)
cam = GradCAM(final_model, gradcam_layer(final_model, ARCHITECTURE))
heatmap, predicted, confidence = cam.generate(tensor)
cam.close()
OUTPUT_DIR.mkdir(exist_ok=True)
heatmap_path = OUTPUT_DIR / f'improved_gradcam_{RUN_ID}.png'
save_overlay(display_image, heatmap, str(heatmap_path))
print(f'Image: {image_path.name} | actual: {image_path.parent.name} | prediction: {CLASS_NAMES[predicted]} ({confidence:.1%})')
display(Image.open(heatmap_path))